In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/spam_preprocessed.csv')

# Drop any rows where cleaning produced empty text
df = df.dropna(subset=['cleaned_message'])
df = df[df['cleaned_message'].str.strip() != '']

print(df.shape)
df[['cleaned_message', 'label_num']].head()

(5150, 5)


,cleaned_message,label_num
0,jurong point crazi avail bugi great world buff...,0
1,lar joke wif oni,0
2,free entri wkli comp win cup final tkt 21st ma...,1
3,dun say earli hor alreadi say,0
4,nah dont think goe usf live around though,0


In [2]:
X = df['cleaned_message']   # input (the text)
y = df['label_num']         # target (0=ham, 1=spam)

print("X shape:", X.shape)
print("y distribution:\n", y.value_counts())

X shape: (5150,)
y distribution:
 label_num
0    4497
1     653
Name: count, dtype: int64


TRAIN/TEST SPILIT - BEFORE VECTORIZE

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% held out for testing
    random_state=42,      # reproducibility
    stratify=y            # preserve class balance in both splits
)

print("Train size:", X_train.shape[0])
print("Test size: ", X_test.shape[0])
print("\nTrain spam %:", y_train.mean() * 100)
print("Test spam %: ", y_test.mean() * 100)

Train size: 4120
Test size:  1030

Train spam %: 12.669902912621358
Test spam %:  12.718446601941746


 fit TF-IDF — on TRAINING data only

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create the vectorizer
tfidf = TfidfVectorizer(max_features=3000)

# FIT on training data, then TRANSFORM it
X_train_tfidf = tfidf.fit_transform(X_train)

# Only TRANSFORM the test data (do NOT fit!)
X_test_tfidf = tfidf.transform(X_test)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape: ", X_test_tfidf.shape)

Train TF-IDF shape: (4120, 3000)
Test TF-IDF shape:  (1030, 3000)


In [5]:
# What does the vocabulary look like?
print("Vocabulary size:", len(tfidf.vocabulary_))
print("\nSample features:", tfidf.get_feature_names_out()[:20])

# The output is a sparse matrix (mostly zeros)
print("\nMatrix type:", type(X_train_tfidf))
print("Density:", X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]) * 100, "% non-zero")

Vocabulary size: 3000

Sample features: ['008704050406' '0089mi' '0121' '01223585236' '01223585334' '0125698789'
 '020603' '0207' '02070836089' '02072069400' '02073162414' '020903' '021'
 '050703' '0578' '060505' '061104' '071104' '0776xxxxxxx' '07xxxxxxxxx']

Matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Density: 0.2304611650485437 % non-zero


In [6]:
# Take the first training message and show its top weighted words
import numpy as np

feature_names = tfidf.get_feature_names_out()
first_msg_vector = X_train_tfidf[0].toarray().flatten()

# Get top 10 highest-weighted words in this message
top_indices = first_msg_vector.argsort()[-10:][::-1]

print("Original:", X_train.iloc[0])
print("\nTop weighted words:")
for idx in top_indices:
    if first_msg_vector[idx] > 0:
        print(f"  {feature_names[idx]}: {first_msg_vector[idx]:.3f}")

Original: gam gone outstand inning

Top weighted words:
  gam: 0.538
  outstand: 0.512
  inning: 0.512
  gone: 0.431
